In [16]:
# azobenzene geometry generator

from collections import deque
from pathlib import Path

from ase import Atoms
from ase.visualize import view
import msgpack
import numpy as np

reference_path = Path("../data/azoflip_data/MSGpack/1367.msgpack")
if not reference_path.is_file():
    raise FileNotFoundError(f"Reference MessagePack does not exist: {reference_path}")
with reference_path.open("rb") as reference_file:
    reference_directory = next(msgpack.Unpacker(reference_file, strict_map_key=False))

E_INCHIKEY = "DMLAVOWQYNRWNQ-BUHFOSPRNA-N"
reference = next((geometry for geometry in reference_directory.values()
                  if geometry["species"]["inchikey"] == E_INCHIKEY), None)
if reference is None:
    raise ValueError(f"No E-azobenzene reference was found in {reference_path}")
atomic_numbers = np.asarray([int(atom[0]) for atom in reference["xyz"]])
reference_positions = np.asarray([atom[1:] for atom in reference["xyz"]], dtype=float)

covalent_radii = {1: 0.31, 6: 0.76, 7: 0.71}
adjacency = [set() for _ in atomic_numbers]
for i in range(len(atomic_numbers)):
    for j in range(i):
        cutoff = 1.25 * (covalent_radii[atomic_numbers[i]] + covalent_radii[atomic_numbers[j]])
        if np.linalg.norm(reference_positions[i] - reference_positions[j]) < cutoff:
            adjacency[i].add(j)
            adjacency[j].add(i)
nitrogen_indices = np.flatnonzero(atomic_numbers == 7).tolist()
if len(nitrogen_indices) != 2:
    raise ValueError(f"Expected two nitrogens, found {nitrogen_indices}")
n1, n2 = nitrogen_indices
if n2 not in adjacency[n1]:
    raise ValueError("Could not identify the N=N bond in the reference geometry.")
adjacency[n1].remove(n2)
adjacency[n2].remove(n1)
fragment_2_set, queue = {n2}, deque([n2])
while queue:
    atom = queue.popleft()
    for neighbour in adjacency[atom]:
        if neighbour not in fragment_2_set:
            fragment_2_set.add(neighbour)
            queue.append(neighbour)
fragment_2 = np.asarray(sorted(fragment_2_set))
fragment_1 = np.asarray(sorted(set(range(len(atomic_numbers))) - fragment_2_set))
if len(fragment_1) != len(fragment_2):
    raise ValueError("Splitting at N=N did not produce two phenyl-N fragments.")

def attached_carbon(nitrogen):
    carbons = [atom for atom in adjacency[nitrogen] if atomic_numbers[atom] == 6]
    if len(carbons) != 1:
        raise ValueError(f"Expected one carbon attached to N index {nitrogen}; found {carbons}")
    return carbons[0]

c1, c2 = attached_carbon(n1), attached_carbon(n2)
ring_2 = fragment_2[fragment_2 != n2]

def rotate_about_axis(points, atom_indices, origin, axis, angle_degrees):
    axis = axis / np.linalg.norm(axis)
    angle = np.radians(angle_degrees)
    vectors = points[atom_indices] - origin
    points[atom_indices] = (vectors * np.cos(angle)
        + np.cross(axis, vectors) * np.sin(angle)
        + np.outer(vectors @ axis, axis) * (1.0 - np.cos(angle)) + origin)

def raw_dihedral_degrees(points, indices):
    p0, p1, p2, p3 = points[list(indices)]
    b0, b1, b2 = p1 - p0, p2 - p1, p3 - p2
    axis = b1 / np.linalg.norm(b1)
    v, w = b0 - np.dot(b0, axis) * axis, b2 - np.dot(b2, axis) * axis
    return float(np.degrees(np.arctan2(np.dot(np.cross(axis, v), w), np.dot(v, w))))

def dihedral_degrees(points, indices):
    """Return the chemical CNNC dihedral: cis=0 degrees and trans=180 degrees."""
    raw_dihedral = raw_dihedral_degrees(points, indices)
    return float(abs(180.0 - abs(raw_dihedral)))

def angular_error(actual, expected):
    return (actual - expected + 180.0) % 360.0 - 180.0

def angle_degrees(points, indices):
    """Return the angle formed by three atom indices in degrees."""
    i, j, k = indices
    first = points[i] - points[j]
    second = points[k] - points[j]
    cosine = np.dot(first, second) / (np.linalg.norm(first) * np.linalg.norm(second))
    return float(np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0))))

def rotate_fragment_to_direction(points, atom_indices, origin, source, target):
    """Rigidly rotate a fragment so `source` points along `target`."""
    source = source / np.linalg.norm(source)
    target = target / np.linalg.norm(target)
    cosine = float(np.clip(np.dot(source, target), -1.0, 1.0))
    axis = np.cross(source, target)
    axis_norm = np.linalg.norm(axis)
    if axis_norm < 1e-12:
        if cosine > 0.0:
            return
        basis = np.eye(3)[np.argmin(np.abs(source))]
        axis = np.cross(source, basis)
        axis_norm = np.linalg.norm(axis)
    rotate_about_axis(
        points, atom_indices, origin, axis / axis_norm, np.degrees(np.arccos(cosine))
    )

def azobenzene_geometry(
    nn_bond_length, nnc_angle_deg, ring_twist_angle_deg, cnnc_dihedral_deg
):
    """Return an ASE Atoms azobenzene geometry with the requested coordinates.

    `nnc_angle_deg` is imposed symmetrically at C1-N1-N2 and N1-N2-C2.
    `ring_twist_angle_deg` rotates the second phenyl ring about the N2-C2 bond.
    The CNNC dihedral follows the C1-N1-N2-C2 convention used above.
    """
    values = np.asarray(
        [nn_bond_length, nnc_angle_deg, ring_twist_angle_deg, cnnc_dihedral_deg],
        dtype=float,
    )
    if not np.all(np.isfinite(values)):
        raise ValueError("All geometry parameters must be finite.")
    if nn_bond_length <= 0.0:
        raise ValueError("nn_bond_length must be positive.")
    if not 0.0 < nnc_angle_deg < 180.0:
        raise ValueError("nnc_angle_deg must be strictly between 0 and 180 degrees.")
    if not 0.0 <= cnnc_dihedral_deg <= 180.0:
        raise ValueError("cnnc_dihedral_deg must be between 0 and 180 degrees (cis=0, trans=180).")

    positions = reference_positions.copy()
    nn_axis = positions[n2] - positions[n1]
    nn_unit = nn_axis / np.linalg.norm(nn_axis)
    positions[fragment_2] += (nn_bond_length - np.linalg.norm(nn_axis)) * nn_unit

    # Reorient each rigid phenyl-N fragment to the requested symmetric NNC angle.
    target_angle = np.radians(nnc_angle_deg)
    c1_vector = positions[c1] - positions[n1]
    c1_perpendicular = c1_vector - np.dot(c1_vector, nn_unit) * nn_unit
    c1_perpendicular /= np.linalg.norm(c1_perpendicular)
    c1_target = np.cos(target_angle) * nn_unit + np.sin(target_angle) * c1_perpendicular
    rotate_fragment_to_direction(positions, fragment_1, positions[n1], c1_vector, c1_target)

    c2_vector = positions[c2] - positions[n2]
    c2_perpendicular = c2_vector - np.dot(c2_vector, -nn_unit) * (-nn_unit)
    c2_perpendicular /= np.linalg.norm(c2_perpendicular)
    c2_target = -np.cos(target_angle) * nn_unit + np.sin(target_angle) * c2_perpendicular
    rotate_fragment_to_direction(positions, fragment_2, positions[n2], c2_vector, c2_target)

    # Rotate fragment 2 around N=N to match the existing signed CNNC convention.
    current_raw_dihedral = raw_dihedral_degrees(positions, (c1, n1, n2, c2))
    candidates = []
    for target_raw_dihedral in (180.0 - cnnc_dihedral_deg, cnnc_dihedral_deg - 180.0):
        for sign in (1.0, -1.0):
            candidate = positions.copy()
            rotate_about_axis(
                candidate, fragment_2, candidate[n2], candidate[n2] - candidate[n1],
                sign * (target_raw_dihedral - current_raw_dihedral),
            )
            error = abs(dihedral_degrees(candidate, (c1, n1, n2, c2)) - cnnc_dihedral_deg)
            candidates.append((error, candidate))
    positions = min(candidates, key=lambda item: item[0])[1]

    before_twist = positions.copy()
    rotate_about_axis(
        positions, ring_2, positions[n2], positions[c2] - positions[n2], ring_twist_angle_deg
    )

    expected_angle = nnc_angle_deg
    checks = (
        np.isclose(np.linalg.norm(positions[n2] - positions[n1]), nn_bond_length, atol=1e-10),
        np.isclose(angle_degrees(positions, (c1, n1, n2)), expected_angle, atol=1e-8),
        np.isclose(angle_degrees(positions, (n1, n2, c2)), expected_angle, atol=1e-8),
        abs(dihedral_degrees(positions, (c1, n1, n2, c2)) - cnnc_dihedral_deg) < 1e-8,
        np.isclose(np.linalg.norm(before_twist[n2] - before_twist[n1]), np.linalg.norm(positions[n2] - positions[n1]), atol=1e-10),
        np.isclose(angle_degrees(before_twist, (c1, n1, n2)), angle_degrees(positions, (c1, n1, n2)), atol=1e-8),
        np.isclose(angle_degrees(before_twist, (n1, n2, c2)), angle_degrees(positions, (n1, n2, c2)), atol=1e-8),
        abs(dihedral_degrees(before_twist, (c1, n1, n2, c2)) - dihedral_degrees(positions, (c1, n1, n2, c2))) < 1e-8,
    )
    if not all(checks):
        raise AssertionError("The requested azobenzene internal coordinates were not preserved.")

    atoms = Atoms(numbers=atomic_numbers, positions=positions)
    atoms.info.update({
        "nn_bond_length_angstrom": float(nn_bond_length),
        "nnc_angle_deg": float(nnc_angle_deg),
        "ring_twist_angle_deg": float(ring_twist_angle_deg),
        "cnnc_dihedral_deg": float(cnnc_dihedral_deg),
        "reference_geometry_id": reference["id"],
    })
    return atoms


In [15]:
# Example: ASE Atoms stores the geometry as element labels and XYZ coordinates.
example_atoms = azobenzene_geometry(1.5, 120.0, 100.0, 180)
for symbol, (x, y, z) in zip(example_atoms.get_chemical_symbols(), example_atoms.positions):
    print(f"{symbol:<2} {x: .8f} {y: .8f} {z: .8f}")
view(example_atoms)


C   4.45729516 -0.77450743  0.27466805
C   3.91040575 -0.49939793  1.48986157
C   2.60164642 -0.12750039  1.55524066
C   1.75967418 -0.01493468  0.47549178
N   0.45230682  0.23035649  0.70560459
N  -0.49508979  0.38918048 -0.44644342
C  -1.83322640  0.64024464 -0.21091484
C  -2.51583961  1.80345846 -0.28225291
C  -3.90759413  1.94848309 -0.02353739
C  -4.63987272  0.77221340  0.40316895
C  -3.92594948 -0.47406047  0.54537643
C  -2.59344732 -0.44718337  0.18481616
C   2.39963697 -0.23826883 -0.82597378
C   3.75451982 -0.64778177 -0.96593731
H   5.48451772 -0.87818400  0.17916255
H   4.56243684 -0.40746665  2.31148742
H   2.22953585  0.06363802  2.54614799
H  -1.91350253  2.74819880 -0.52778181
H  -4.52830411  2.75780760 -0.21201241
H  -5.67807196  0.65375160  0.75515691
H  -4.24086225 -1.35404820  0.92920533
H  -1.98593662 -1.40664025  0.28156826
H   1.79617158 -0.10943424 -1.73029137
H   4.16404590 -0.96678668 -1.87019880


<Popen: returncode: None args: ['/home/lim_yt/micromamba/envs/xmace311/bin/p...>

In [17]:
# Generate the full parametric azobenzene grid as an unlabeled multi-frame XYZ file.
# This cell requires only the standalone azobenzene_geometry cell above.

from itertools import product

import ase.io

nn_bond_lengths_angstrom = np.round(np.arange(1.20, 1.50 + 0.001, 0.05), 2)
nnc_angles_deg = np.arange(110.0, 140.0 + 0.1, 5.0)
ring_twist_angles_deg = 0.1 + np.arange(11, dtype=float) * 17.98
cnnc_dihedrals_deg = 0.1 + np.arange(21, dtype=float) * 8.99

output_path = Path("../data/azoflip_data/XYZ/azobenzene_grid.xyz")
parameter_grid = product(
    nn_bond_lengths_angstrom,
    nnc_angles_deg,
    ring_twist_angles_deg,
    cnnc_dihedrals_deg,
)
expected_frame_count = (
    len(nn_bond_lengths_angstrom)
    * len(nnc_angles_deg)
    * len(ring_twist_angles_deg)
    * len(cnnc_dihedrals_deg)
)
assert expected_frame_count == 11_319

output_path.parent.mkdir(parents=True, exist_ok=True)
frame_count = 0
with output_path.open("w") as xyz_file:
    for nn_bond_length, nnc_angle, ring_twist_angle, cnnc_dihedral in parameter_grid:
        atoms = azobenzene_geometry(
            nn_bond_length, nnc_angle, ring_twist_angle, cnnc_dihedral
        )
        xyz_file.write(f"{len(atoms)}\n")
        xyz_file.write(
            f"nn_bond_length_angstrom={nn_bond_length:.8f} "
            f"nnc_angle_deg={nnc_angle:.8f} "
            f"ring_twist_angle_deg={ring_twist_angle:.8f} "
            f"cnnc_dihedral_deg={cnnc_dihedral:.8f} "
            f"reference_geometry_id={atoms.info['reference_geometry_id']}\n"
        )
        for symbol, (x, y, z) in zip(atoms.get_chemical_symbols(), atoms.positions):
            xyz_file.write(f"{symbol:<2} {x: .8f} {y: .8f} {z: .8f}\n")
        frame_count += 1

assert frame_count == expected_frame_count

first_frame = ase.io.read(output_path, index=0)
last_frame = ase.io.read(output_path, index=-1)
for frame in (first_frame, last_frame):
    assert len(frame) == 24
assert np.isclose(first_frame.info["nn_bond_length_angstrom"], nn_bond_lengths_angstrom[0])
assert np.isclose(first_frame.info["nnc_angle_deg"], nnc_angles_deg[0])
assert np.isclose(first_frame.info["ring_twist_angle_deg"], ring_twist_angles_deg[0])
assert np.isclose(first_frame.info["cnnc_dihedral_deg"], cnnc_dihedrals_deg[0])
assert np.isclose(last_frame.info["nn_bond_length_angstrom"], nn_bond_lengths_angstrom[-1])
assert np.isclose(last_frame.info["nnc_angle_deg"], nnc_angles_deg[-1])
assert np.isclose(last_frame.info["ring_twist_angle_deg"], ring_twist_angles_deg[-1])
assert np.isclose(last_frame.info["cnnc_dihedral_deg"], cnnc_dihedrals_deg[-1])

print(f"Wrote {frame_count:,} geometries to {output_path}")


Wrote 7,986 geometries to ../data/azoflip_data/XYZ/azobenzene_grid.xyz


In [ ]:
# Read directory.msgpack

import msgpack

directory_path = "../data/azoflip_data/MSGpack/directory.msgpack"
with open(directory_path, "rb") as directory_file:
    directory = next(msgpack.Unpacker(directory_file, strict_map_key=False))


In [ ]:
# Generate an azobenzene CNNC torsion / N--N distance grid.
# `directory` must be loaded first (for example, by the preceding MessagePack cell).

from collections import deque
from pathlib import Path

from ase import Atoms
import numpy as np

# Both endpoints are included.
cnnc_dihedrals_deg = np.arange(0.0, 180.0 + 0.5, 5.0)
nn_distances_angstrom = np.linspace(1.0, 1.6, 31)  # 0.02 Angstrom spacing
output_path = Path("../data/azoflip_data/XYZ/azobenzene_cnnd_nn_grid.xyz")

E_INCHIKEY = "DMLAVOWQYNRWNQ-BUHFOSPRNA-N"
reference = next(
    geometry
    for geometry in directory.values()
    if geometry["species"]["inchikey"] == E_INCHIKEY
)
atomic_numbers = np.asarray([int(atom[0]) for atom in reference["xyz"]])
reference_positions = np.asarray([atom[1:] for atom in reference["xyz"]], dtype=float)
symbols = {1: "H", 6: "C", 7: "N"}

# Infer the molecular graph from the reference geometry, then split it at N=N.
covalent_radii = {1: 0.31, 6: 0.76, 7: 0.71}
adjacency = [set() for _ in atomic_numbers]
for i in range(len(atomic_numbers)):
    for j in range(i):
        cutoff = 1.25 * (covalent_radii[atomic_numbers[i]] + covalent_radii[atomic_numbers[j]])
        if np.linalg.norm(reference_positions[i] - reference_positions[j]) < cutoff:
            adjacency[i].add(j)
            adjacency[j].add(i)

nitrogen_indices = np.flatnonzero(atomic_numbers == 7).tolist()
if len(nitrogen_indices) != 2:
    raise ValueError(f"Expected two nitrogens, found {nitrogen_indices}")
n1, n2 = nitrogen_indices
if n2 not in adjacency[n1]:
    raise ValueError("Could not identify the N=N bond in the reference geometry.")

adjacency[n1].remove(n2)
adjacency[n2].remove(n1)
fragment_2 = set([n2])
queue = deque([n2])
while queue:
    atom = queue.popleft()
    for neighbour in adjacency[atom]:
        if neighbour not in fragment_2:
            fragment_2.add(neighbour)
            queue.append(neighbour)
if len(fragment_2) * 2 != len(atomic_numbers):
    raise ValueError("Splitting at N=N did not produce two phenyl-N fragments.")
fragment_2 = np.asarray(sorted(fragment_2))
fragment_1 = np.asarray(sorted(set(range(len(atomic_numbers))) - set(fragment_2)))

def attached_carbon(nitrogen, other_nitrogen):
    carbons = [atom for atom in adjacency[nitrogen] if atomic_numbers[atom] == 6]
    if len(carbons) != 1:
        raise ValueError(f"Expected one carbon attached to N index {nitrogen}; found {carbons}")
    return carbons[0]

c1 = attached_carbon(n1, n2)
c2 = attached_carbon(n2, n1)
ring_1_carbons = fragment_1[atomic_numbers[fragment_1] == 6]
ring_2_carbons = fragment_2[atomic_numbers[fragment_2] == 6]
BENZENE_RING_1_ATOMS = tuple(int(index) for index in ring_1_carbons)
BENZENE_RING_2_ATOMS = tuple(int(index) for index in ring_2_carbons)
ring_2 = fragment_2[fragment_2 != n2]

def dihedral_degrees(points, indices):
    p0, p1, p2, p3 = points[list(indices)]
    b0, b1, b2 = p1 - p0, p2 - p1, p3 - p2
    axis = b1 / np.linalg.norm(b1)
    v = b0 - np.dot(b0, axis) * axis
    w = b2 - np.dot(b2, axis) * axis
    return np.degrees(np.arctan2(np.dot(np.cross(axis, v), w), np.dot(v, w)))

def rotate_about_axis(points, atom_indices, origin, axis, angle_degrees):
    axis = axis / np.linalg.norm(axis)
    angle = np.radians(angle_degrees)
    vectors = points[atom_indices] - origin
    rotated = (
        vectors * np.cos(angle)
        + np.cross(axis, vectors) * np.sin(angle)
        + np.outer(vectors @ axis, axis) * (1.0 - np.cos(angle))
    )
    points[atom_indices] = rotated + origin

def angular_error(actual, expected):
    return (actual - expected + 180.0) % 360.0 - 180.0

def ring_normal(points, ring_carbons):
    _, _, right_vectors = np.linalg.svd(points[ring_carbons] - points[ring_carbons].mean(axis=0))
    return right_vectors[-1]

def benzene_ring_normal_angle(atoms, ring_1=BENZENE_RING_1_ATOMS, ring_2=BENZENE_RING_2_ATOMS):
    """Return the unsigned angle (0--90 degrees) between two best-fit benzene-ring normals."""
    def plane_normal(atom_indices):
        positions = atoms.get_positions()[list(atom_indices)]
        _, singular_values, right_vectors = np.linalg.svd(positions - positions.mean(axis=0))
        if singular_values[1] < 1e-12:
            raise ValueError("Cannot determine a ring normal from collinear atom positions.")
        return right_vectors[-1]

    normal_1, normal_2 = plane_normal(ring_1), plane_normal(ring_2)
    cosine = np.clip(abs(np.dot(normal_1, normal_2)), -1.0, 1.0)
    return float(np.degrees(np.arccos(cosine)))

def normal_angle_degrees(points):
    return benzene_ring_normal_angle(Atoms(numbers=atomic_numbers, positions=points))

def apply_linear_nc_ring_torsion(points, cnnc_dihedral):
    """Rotate only the second phenyl ring about N2--C2 by -CNNC / 2."""
    nc_ring_torsion = -0.5 * cnnc_dihedral
    rotated = points.copy()
    n2_position = rotated[n2].copy()
    rotate_about_axis(rotated, ring_2, rotated[n2], rotated[c2] - rotated[n2], nc_ring_torsion)
    assert np.allclose(rotated[n2], n2_position, atol=1e-12)
    return rotated, nc_ring_torsion

def constrained_geometry(nn_distance, cnnc_dihedral):
    positions = reference_positions.copy()
    axis = positions[n2] - positions[n1]
    positions[fragment_2] += (nn_distance - np.linalg.norm(axis)) * axis / np.linalg.norm(axis)
    current = dihedral_degrees(positions, (c1, n1, n2, c2))
    rotation = cnnc_dihedral - current

    # Select the rotation sign that matches this dihedral convention.
    candidates = []
    for sign in (1.0, -1.0):
        candidate = positions.copy()
        rotate_about_axis(candidate, fragment_2, candidate[n2], candidate[n2] - candidate[n1], sign * rotation)
        error = abs(angular_error(dihedral_degrees(candidate, (c1, n1, n2, c2)), cnnc_dihedral))
        candidates.append((error, candidate))
    positions = min(candidates, key=lambda item: item[0])[1]
    positions, nc_ring_torsion = apply_linear_nc_ring_torsion(positions, cnnc_dihedral)

    measured_distance = np.linalg.norm(positions[n2] - positions[n1])
    measured_dihedral = dihedral_degrees(positions, (c1, n1, n2, c2))
    assert np.isclose(measured_distance, nn_distance, atol=1e-10)
    assert abs(angular_error(measured_dihedral, cnnc_dihedral)) < 1e-8
    assert np.isfinite(normal_angle_degrees(positions))
    assert np.isclose(nc_ring_torsion, -0.5 * cnnc_dihedral, atol=1e-12)
    return positions

output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w") as xyz_file:
    for nn_distance in nn_distances_angstrom:
        for cnnc_dihedral in cnnc_dihedrals_deg:
            positions = constrained_geometry(nn_distance, cnnc_dihedral)
            measured_normal_angle = normal_angle_degrees(positions)
            nc_ring_torsion = -0.5 * cnnc_dihedral
            xyz_file.write(f"{len(atomic_numbers)}\n")
            xyz_file.write(
                f"cnnc_dihedral_deg={cnnc_dihedral:.8f} nn_distance_angstrom={nn_distance:.8f} "
                f"measured_ring_normal_angle_deg={measured_normal_angle:.8f} "
                f"nc_ring_torsion_deg={nc_ring_torsion:.8f} reference_geometry_id={reference['id']}\n"
            )
            for atomic_number, (x, y, z) in zip(atomic_numbers, positions):
                xyz_file.write(f"{symbols[atomic_number]:<2} {x: .8f} {y: .8f} {z: .8f}\n")

print(f"Wrote {len(cnnc_dihedrals_deg) * len(nn_distances_angstrom):,} geometries to {output_path} ({cnnc_dihedrals_deg[0]:g}--{cnnc_dihedrals_deg[-1]:g} degrees; {nn_distances_angstrom[0]:.2f}--{nn_distances_angstrom[-1]:.2f} Angstrom).")
print("Applied a linear N2--C2 ring torsion from 0 to -90 degrees.")


Wrote 1,147 geometries to ../data/azoflip_data/XYZ/azobenzene_cnnd_nn_grid.xyz (0--180 degrees; 1.00--1.60 Angstrom).


In [ ]:
# Predict state-resolved energies and forces for the CNNC / N--N grid.
# The model was saved with CUDA tensors, so execute this cell on a CUDA-enabled host.

import json
from pathlib import Path

import ase.io
import numpy as np
import torch
from mace.calculators import MACECalculator

grid_path = Path("../data/azoflip_data/XYZ/azobenzene_cnnd_nn_grid.xyz")
model_path = Path("../outputs/base_models/base_model_azoflip.pt")
if not grid_path.is_file():
    raise FileNotFoundError(f"Grid XYZ does not exist: {grid_path}")
if not model_path.is_file():
    raise FileNotFoundError(f"Model does not exist: {model_path}")
if not torch.cuda.is_available():
    raise RuntimeError("base_model_azoflip.pt is CUDA-serialized; run this cell with a CUDA-enabled PyTorch installation.")

device = "cuda"
calculator = MACECalculator(model_paths=model_path, device=device)
calculator.model.eval()
frames = ase.io.read(grid_path, index=":")
if not frames:
    raise ValueError(f"No structures found in {grid_path}")

predictions = []
for frame_index, atoms in enumerate(frames, start=1):
    atoms.calc = calculator
    energies = np.asarray(atoms.get_potential_energy(), dtype=float).reshape(1, -1)
    forces = np.asarray(atoms.get_forces(), dtype=float)
    if forces.shape != (len(atoms), energies.shape[1], 3):
        raise ValueError(
            f"Unexpected force shape for frame {frame_index}: {forces.shape}; "
            f"expected {(len(atoms), energies.shape[1], 3)}"
        )
    predictions.append((energies, forces))
    if frame_index % 100 == 0 or frame_index == len(frames):
        print(f"Predicted {frame_index:,}/{len(frames):,} structures")

# Use a temporary file so the existing grid is retained if prediction fails.
temporary_path = grid_path.with_suffix(grid_path.suffix + ".tmp")
with temporary_path.open("w") as xyz_file:
    for atoms, (energies, forces) in zip(frames, predictions):
        cnnc_dihedral = float(atoms.info["cnnc_dihedral_deg"])
        nn_distance = float(atoms.info["nn_distance_angstrom"])
        reference_id = atoms.info["reference_geometry_id"]
        xyz_file.write(f"{len(atoms)}\n")
        xyz_file.write(
            "Properties=species:S:1:pos:R:3 "
            f'REF_energy="_JSON {json.dumps(energies.tolist())}" ' 
            f'REF_forces="_JSON {json.dumps(forces.tolist())}" ' 
            f"cnnc_dihedral_deg={cnnc_dihedral:.8f} "
            f"nn_distance_angstrom={nn_distance:.8f} "
            f"reference_geometry_id={reference_id}\n"
        )
        for symbol, (x, y, z) in zip(atoms.get_chemical_symbols(), atoms.positions):
            xyz_file.write(f"{symbol:<2} {x: .8f} {y: .8f} {z: .8f}\n")
temporary_path.replace(grid_path)
print(f"Wrote {len(frames):,} predicted structures to {grid_path}")


In [ ]:
# Generate the same CNNC / N--N grid after rotating one phenyl ring by 90 degrees.
# Run the `generate-cnnd-nn-grid` cell first: this cell reuses its reference and geometry helpers.

import json

ring_rotation_deg = 90.0
rotated_output_path = Path("../data/azoflip_data/XYZ/azobenzene_cnnd_nn_grid_ring_rotated_90.xyz")
ring_2 = fragment_2[fragment_2 != n2]  # Phenyl ring attached to N index n2; N itself remains fixed.
if c2 not in ring_2:
    raise ValueError("The selected phenyl-ring fragment does not contain its N--C attachment atom.")

def constrained_geometry_with_rotated_ring(nn_distance, cnnc_dihedral):
    positions = constrained_geometry(nn_distance, cnnc_dihedral)
    # Rotate the complete phenyl ring about its N--C bond. C2 lies on the axis.
    rotate_about_axis(
        positions, ring_2, positions[n2], positions[c2] - positions[n2], ring_rotation_deg
    )
    assert np.isclose(np.linalg.norm(positions[n2] - positions[n1]), nn_distance, atol=1e-10)
    assert abs(angular_error(dihedral_degrees(positions, (c1, n1, n2, c2)), cnnc_dihedral)) < 1e-8
    return positions

rotated_output_path.parent.mkdir(parents=True, exist_ok=True)
with rotated_output_path.open("w") as xyz_file:
    for nn_distance in nn_distances_angstrom:
        for cnnc_dihedral in cnnc_dihedrals_deg:
            positions = constrained_geometry_with_rotated_ring(nn_distance, cnnc_dihedral)
            xyz_file.write(f"{len(atomic_numbers)}\n")
            xyz_file.write(
                f"cnnc_dihedral_deg={cnnc_dihedral:.8f} nn_distance_angstrom={nn_distance:.8f} "
                f"benzene_ring_rotation_deg={ring_rotation_deg:.8f} reference_geometry_id={reference['id']}\n"
            )
            for atomic_number, (x, y, z) in zip(atomic_numbers, positions):
                xyz_file.write(f"{symbols[atomic_number]:<2} {x: .8f} {y: .8f} {z: .8f}\n")

print(f"Wrote {len(cnnc_dihedrals_deg) * len(nn_distances_angstrom):,} 90-degree ring-rotated geometries to {rotated_output_path}")
